# PhageScope Pipeline

Basic packages 

The PhageScope data are downloaded via snakemake and reports generated on up to 50k randomly selected datapoint. 




## File list



In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

data_dir = "../data"


### Protein Metadata

Temporary creation of big lists for later use. 

In [2]:
# Accumulators
chunk_num = 0
total_rows = 0
phage_ids = []
protein_ids = []
source = [] # Source of the protein, e.g. DDBJ, CHVD, etc. match the directory names for the fasta files


for chunk in pd.read_csv(os.path.join(data_dir, "merged/merged_annotated_proteins_metadata.csv"), chunksize=10000):
    total_rows += len(chunk)
    phage_ids.extend(chunk['Phage_ID'].tolist())
    protein_ids.extend(chunk['Protein_ID'].tolist())
    source.extend(chunk['Phage_source'].tolist())


    # ------------------------------
    chunk_num += 1
    if chunk_num >= 55:  # Limit to 5 chunks for demonstration
        break

In [3]:
print(pd.Series(source).unique())

['RefSeq' 'Genbank']


In [4]:

#transcription_terminator_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_transcription_terminator_metadata.csv"))
#phage_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_metadata.csv"))
#phage_protein_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_protein_metadata.csv"))

## Merging protein and phage Fasta

Some sources have one fasta file per protein, other have one fasta file containing all proteins. 

In total, there are more than 23 milion fasta files (just protein !) Some sources have one file containing multiple sequences, while other sources have one fasta file per sequence. 

```
Unique sources: 13
RefSeq: 480642 FASTA files
IGVD: 1 FASTA files
PhagesDB: 351898 FASTA files
GPD: 7616044 FASTA files
GOV2: 1 FASTA files
CHVD: 1 FASTA files
MGV: 10517011 FASTA files
DDBJ: 17391 FASTA files
STV: 1 FASTA files
EMBL: 11095 FASTA files
TemPhD: 3465586 FASTA files
GVD: 746146 FASTA files
Genbank: 217128 FASTA files
Total fasta files found: 23422945
```

To obtain the sequence from the protein ID, we should be able to query the sequences at will and for this, having a coeherent database is vital. We will then merge the sequences in one file per source which should allow later to create a performant database, which is not possible with 23 million files. 

The structure is like this: 

```
protein_fasta/
  └── source1/
        ├── source1/
        │     ├── phageA/
        │     │     ├── file1.fasta
        │     │     └── file2.fasta
        │     └── phageB/
        │           └── file3.fasta
  └── source2/
        └── single.fasta

```

### Merging Fasta files per source

Using the script `merge_protein_fasta.py` with the snakemake rules `snakemake --cores 2 data/protein_fasta_merged/DDBJ.fasta` for each file, it will merge the files if necessary into data/protein_fasta_merged. 





Visualize with `snakemake --cores 4 --dag | dot -Tsvg > dag_all.svg`

More complete command with cache and logging: `snakemake --cache --printshellcmds --reason --timestamp --cores 8 `

## Activation du cache pour les règles lourdes

Créer les répertoires nécessaires: 

```
sudo mkdir -p /mnt/snakemake-cache
sudo chown $(whoami) /mnt/snakemake-cache

```

`export SNAKEMAKE_OUTPUT_CACHE=/mnt/snakemake-cache/`

Le caching est activé pour les règles de téléchargement. 

Afin d'exécuter les scripts dans le bon environnement, on export l'environement conda avec `conda env export > envs/base_env.yaml`

On exécute snakemake directement et on donne les arguments conda nécessaires (exemple avec merge_annotated_proteins_metadata_tsvs): 

`snakemake all --cache --use-conda --conda-frontend mamba --printshellcmds --cores 2`

Attention: certain crashes peuvent être dûs à une surcharge I/O en cas de multithreading trop important. 



# Phage Metadata Exploration

Explore the data for further pip

In [5]:
import pandas as pd
import numpy as np
import os


In [6]:
# Read data/merged/merged_phage_metadata.csv
phage_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_metadata.csv"))
annotated_proteins_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_annotated_proteins_metadata.csv"))
phage_anti_crispr_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_anti_crispr_metadata.csv"))
phage_transmembrane_protein_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_transmembrane_protein_metadata.csv"))
phage_trna_tmrna_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_trna_tmrna_metadata.csv"))
phage_virulent_factor_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_virulent_factor_metadata.csv"))
transcription_terminator_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_transcription_terminator_metadata.csv"))


# For each Phage_source, get a sample of 10 phages in a new dataframe
sampled_phages = phage_metadata.groupby('Phage_source').apply(lambda x: x.sample(n=10, random_state=42)).reset_index(drop=True)

/tmp/ipykernel_11072/2354234772.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  phage_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_phage_metadata.csv"))
/tmp/ipykernel_11072/2354234772.py:8: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  transcription_terminator_metadata = pd.read_csv(os.path.join(data_dir, "merged/merged_transcription_terminator_metadata.csv"))
/tmp/ipykernel_11072/2354234772.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_phages = phage_metadata.groupby('Phage_source').apply(lambda x: x.sample(n=10, random_state=42)).reset_index(drop=T

In [7]:
# Filter to keep only phage id and phage source
tmp = sampled_phages[['Phage_ID', 'Phage_source']]

# Get only rows with source PhagesDB
tmp[tmp['Phage_source'] == 'PhagesDB']

,Phage_ID,Phage_source


In [8]:
phage_metadata

,Phage_ID,Length,GC_content,Taxonomy,Completeness,Host,Lifestyle,Cluster,Subcluster,Source_DB,Phage_source
0,NC_001330.1,6087,45.178249,Microviridae,NaN,Escherichia coli,virulent,cluster_158413,subcluster_199459,RefSeq,NaN
1,NC_001331.1,7349,61.491359,Inoviridae,NaN,Pseudomonas aeruginosa,temperate,cluster_67555,subcluster_85085,RefSeq,NaN
2,NC_001332.1,6744,42.719454,Inoviridae,NaN,Escherichia coli,virulent,cluster_63254,subcluster_79716,RefSeq,NaN
3,NC_001335.1,52297,62.257873,Caudovirales,NaN,Mycobacterium smegmatis,temperate,cluster_272329,subcluster_342024,RefSeq,NaN
4,NC_001341.1,4491,33.288800,NaN,NaN,Acholeplasma laidlawii,virulent,cluster_229441,subcluster_288779,RefSeq,NaN
...,...,...,...,...,...,...,...,...,...,...,...
873713,biochar_6173,10017,65.768194,Caudoviricetes,NaN,Pseudomonas fluorescens,virulent,cluster_402067,subcluster_485570,STV,STV
873714,biochar_6175,10016,63.837859,Caudoviricetes,NaN,Klebsiella pneumoniae,virulent,cluster_66332,subcluster_80652,STV,STV
873715,biochar_6178,10011,65.168315,Caudoviricetes,NaN,Mycobacterium tuberculosis,temperate,cluster_292651,subcluster_353604,STV,STV
873716,biochar_6180,10008,59.102718,Caudoviricetes,NaN,Sinorhizobium meliloti,virulent,cluster_60145,subcluster_73232,STV,STV


In [9]:
# For all phages in phage_metadata, get the repartition of Host
host_repartition = phage_metadata['Host'].value_counts(normalize=True) * 100
host_repartition = host_repartition.reset_index()
host_repartition.columns = ['Host', 'Percentage']
print(host_repartition.head(400))

                                           Host  Percentage
0                           Salmonella enterica    5.482056
1                      Lawsonia intracellularis    4.388094
2                      Clostridioides difficile    3.368986
3                               Lachnospiraceae    3.176244
4                                   Bacteroides    2.557042
..                                          ...         ...
395                               Eubacterium_I    0.020259
396                               Eubacterium_F    0.020259
397                                    4C28d-15    0.020259
398                           Erwinia amylovora    0.020259
399  Bacteroides B dorei;Bacteroides B vulgatus    0.020259

[400 rows x 2 columns]


In [10]:
sampled_phages

,Phage_ID,Length,GC_content,Taxonomy,Completeness,Host,Lifestyle,Cluster,Subcluster,Source_DB,Phage_source
0,SRS143070_a1_ct78652,41502,51.240904,Siphoviridae,NaN,Rhodanobacter,virulent,cluster_199050,subcluster_240157,CHVD,CHVD
1,SRS047126_a1_ct151_vs1,43900,43.200456,Siphoviridae,NaN,Aggregatibacter,virulent,cluster_183566,subcluster_221512,CHVD,CHVD
2,SRS148363_a1_ct32938_vs01,15525,42.344605,Caudovirales,NaN,Streptococcus,virulent,cluster_404679,subcluster_488716,CHVD,CHVD
3,SAMN10080906_a1_ct1149,6590,45.402124,Caudovirales,NaN,Clostridiales,virulent,cluster_68414,subcluster_83140,CHVD,CHVD
4,SAMN04261392_a1_ct225484_vs1,27150,46.243094,Siphoviridae,NaN,Lactobacillus,virulent,cluster_470454,subcluster_566398,CHVD,CHVD
5,SAMEA1906534_a1_ct119387_vs1,31606,43.814466,Siphoviridae,NaN,Eubacterium,virulent,cluster_436901,subcluster_527037,CHVD,CHVD
6,SRS049959_a1_ct43923_vs01,47674,56.118639,Caudovirales,NaN,-,temperate,cluster_275305,subcluster_332241,CHVD,CHVD
7,SAMN11031281_a1_ct2909,39228,51.157337,Myoviridae,NaN,Salmonella,temperate,cluster_219025,subcluster_264296,CHVD,CHVD
8,SAMN00791970_a1_ct62912_vs1,13289,52.080668,Siphoviridae,NaN,-,temperate,cluster_443173,subcluster_534388,CHVD,CHVD
9,SRS017757_a1_ct45722_vs01,25295,47.898794,Caudovirales,NaN,Neisseria,virulent,cluster_203363,subcluster_245323,CHVD,CHVD


In [11]:
NUMERICAL_COLUMNS = ['Start', 'Stop', 'Molecular_weight', 'Aromaticity',
       'Instability_index', 'Isoelectric_point', 'Helix_fraction', 'Turn_fraction',
       'Sheet_fraction', 'Reduced_coefficient', 'Oxidized_coefficient']

# get columns types

annotated_proteins_metadata


,Phage_ID,Protein_source,Function_prediction_source,Start,Stop,Strand,Protein_ID,Product,Protein_classification,Molecular_weight,Aromaticity,Instability_index,Isoelectric_point,Helix_fraction,Turn_fraction,Sheet_fraction,Reduced_coefficient,Oxidized_coefficient,Phage_source,Source_DB
0,NC_001330.1,RefSeq,RefSeq,759,2243,+,NP_039590.1,replication initiation protein,replication;,407.4640,0.250000,17.125000,8.750052,0.250000,0.500000,0.000000,0,0,RefSeq,RefSeq
1,NC_001330.1,RefSeq,RefSeq,1805,2158,+,NP_039591.1,internal scaffolding protein,assembly;,5650.1306,0.234043,17.321489,7.018615,0.297872,0.212766,0.170213,9970,9970,RefSeq,RefSeq
2,NC_001330.1,RefSeq,RefSeq,2158,2319,+,NP_039592.1,hypothetical protein,hypothetical;,6235.2870,0.037736,37.058491,6.816041,0.339623,0.132075,0.377358,0,125,RefSeq,RefSeq
3,NC_001330.1,RefSeq,RefSeq,2270,2476,+,NP_039593.1,DNA maturation protein,infection;,8295.3801,0.161765,32.102941,9.300419,0.338235,0.132353,0.279412,22460,22460,RefSeq,RefSeq
4,NC_001330.1,RefSeq,RefSeq,2503,2955,+,NP_039594.1,external scaffolding protein,assembly;,1183.4075,0.000000,15.510000,11.999968,0.200000,0.200000,0.300000,0,0,RefSeq,RefSeq
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23424954,TemPhD_cluster_9999,prodigal,eggNOG-mapper,67870,68043,-,TemPhD_cluster_9999_73,Putative transposase of IS4/5 family (DUF4096),integration;,7048.8755,0.140351,84.473684,9.769168,0.298246,0.210526,0.192982,7450,7450,TemPhD,TemPhD
23424955,TemPhD_cluster_9999,prodigal,eggNOG-mapper,68324,68755,+,TemPhD_cluster_9999_74,Transcriptional regulator,regulation;,350.3913,0.000000,6.666667,5.518123,0.000000,0.000000,0.000000,0,0,TemPhD,TemPhD
23424956,TemPhD_cluster_9999,prodigal,eggNOG-mapper,69006,69329,+,TemPhD_cluster_9999_75,P-type ATPase,unsorted;,4378.1752,0.108108,46.389189,11.451600,0.351351,0.243243,0.243243,1490,1490,TemPhD,TemPhD
23424957,TemPhD_cluster_9999,prodigal,eggNOG-mapper,69185,69865,-,TemPhD_cluster_9999_76,Transposase,integration;,1679.0526,0.062500,53.450000,5.994263,0.375000,0.250000,0.312500,0,0,TemPhD,TemPhD


In [12]:
# Look for specific string in annotated_proteins_metadata Protein_ID column

annotated_proteins_metadata[annotated_proteins_metadata['Protein_ID'].str.contains('Mycobacterium_phage_Scamp')]

,Phage_ID,Protein_source,Function_prediction_source,Start,Stop,Strand,Protein_ID,Product,Protein_classification,Molecular_weight,Aromaticity,Instability_index,Isoelectric_point,Helix_fraction,Turn_fraction,Sheet_fraction,Reduced_coefficient,Oxidized_coefficient,Phage_source,Source_DB
1070445,Mycobacterium_phage_Scamp,prodigal,Iterative search,803,1234,+,Mycobacterium_phage_Scamp_1,minor tail protein,infection;,318.3296,0.000000,153.133333,9.750021,0.000000,0.666667,0.000000,0,0,PhagesDB,PhagesDB
1070446,Mycobacterium_phage_Scamp,prodigal,Iterative search,1326,1640,+,Mycobacterium_phage_Scamp_2,minor tail protein,infection;,3619.1284,0.058824,29.264706,4.407375,0.352941,0.205882,0.352941,5500,5500,PhagesDB,PhagesDB
1070447,Mycobacterium_phage_Scamp,prodigal,eggNOG-mapper,1645,2718,+,Mycobacterium_phage_Scamp_3,N-acetylmuramoyl-L-alanine amidase activity,unsorted;,797.9371,0.285714,48.515714,5.184876,0.714286,0.142857,0.000000,1490,1490,PhagesDB,PhagesDB
1070448,Mycobacterium_phage_Scamp,prodigal,Iterative search,2722,3384,+,Mycobacterium_phage_Scamp_4,minor tail protein,infection;,1205.3681,0.200000,-7.980000,10.835024,0.400000,0.100000,0.200000,6990,6990,PhagesDB,PhagesDB
1070449,Mycobacterium_phage_Scamp,prodigal,-,3363,3755,+,Mycobacterium_phage_Scamp_5,unknown,unsorted;,7292.0426,0.166667,91.390000,5.276557,0.283333,0.216667,0.250000,15470,15470,PhagesDB,PhagesDB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1070522,Mycobacterium_phage_Scamp,prodigal,eggNOG-mapper,48415,48591,-,Mycobacterium_phage_Scamp_78,hypothetical protein,hypothetical;,6704.6497,0.068966,52.005172,4.829575,0.275862,0.120690,0.310345,6990,6990,PhagesDB,PhagesDB
1070523,Mycobacterium_phage_Scamp,prodigal,Iterative search,48593,48805,-,Mycobacterium_phage_Scamp_79,Pfam:GP88,assembly;,7533.2913,0.057143,28.050000,4.050028,0.342857,0.228571,0.285714,12490,12615,PhagesDB,PhagesDB
1070524,Mycobacterium_phage_Scamp,prodigal,-,48897,49103,-,Mycobacterium_phage_Scamp_80,unknown,unsorted;,7088.0232,0.014706,37.127941,6.901470,0.279412,0.250000,0.235294,0,0,PhagesDB,PhagesDB
1070525,Mycobacterium_phage_Scamp,prodigal,Iterative search,49126,50010,-,Mycobacterium_phage_Scamp_81,site-specific recombination directionality fac...,integration;,1578.7697,0.142857,3.221429,9.994292,0.214286,0.142857,0.214286,5500,5500,PhagesDB,PhagesDB


In [12]:
phage_anti_crispr_metadata

,Phage_ID,Protein_ID,Source,Phage_source,Source_DB
0,NC_061433.1,YP_010299115.1,Alignment;Acranker,RefSeq,RefSeq
1,NC_005178.1,NP_938237.1,Alignment;Acranker,RefSeq,RefSeq
2,NC_024387.1,YP_009044826.1,Alignment;Acranker,RefSeq,RefSeq
3,NC_061436.1,YP_010299280.1,Alignment;Acranker,RefSeq,RefSeq
4,NC_041851.1,YP_009591718.1,Alignment;Acranker,RefSeq,RefSeq
...,...,...,...,...,...
299827,biochar_855,biochar_855_95,Acranker,STV,STV
299828,biochar_4575,biochar_4575_23,Acranker,STV,STV
299829,biochar_826,biochar_826_1,Acranker,STV,STV
299830,biochar_2844,biochar_2844_23,Acranker,STV,STV


In [17]:
print("Columns:", phage_anti_crispr_metadata.columns.tolist())
print("\nData types:\n", phage_anti_crispr_metadata.dtypes)
print("\nSample data:\n", phage_anti_crispr_metadata.head())
print("\nNull counts:\n", phage_anti_crispr_metadata.isnull().sum())
print("\nUnique Phage_IDs:", phage_anti_crispr_metadata['Phage_ID'].nunique())

Columns: ['Phage_ID', 'Protein_ID', 'Source', 'Phage_source', 'Source_DB']

Data types:
 Phage_ID        object
Protein_ID      object
Source          object
Phage_source    object
Source_DB       object
dtype: object

Sample data:
       Phage_ID      Protein_ID              Source Phage_source Source_DB
0  NC_061433.1  YP_010299115.1  Alignment;Acranker       RefSeq    RefSeq
1  NC_005178.1     NP_938237.1  Alignment;Acranker       RefSeq    RefSeq
2  NC_024387.1  YP_009044826.1  Alignment;Acranker       RefSeq    RefSeq
3  NC_061436.1  YP_010299280.1  Alignment;Acranker       RefSeq    RefSeq
4  NC_041851.1  YP_009591718.1  Alignment;Acranker       RefSeq    RefSeq

Null counts:
 Phage_ID        0
Protein_ID      0
Source          0
Phage_source    0
Source_DB       0
dtype: int64

Unique Phage_IDs: 124366


In [13]:
phage_transmembrane_protein_metadata

,Phage_ID,Protein_ID,Length,PredictedTMHsNumber,ExpnumberofAAsinTMHs,Expnumberfirst60AAs,TotalprobofNin,POSSIBLENterm,Insidesource,Insidestart,Insideend,TMhelixsource,TMhelixstart,TMhelixend,Outsidesource,Outsidestart,Outsideend,Source_DB
0,NC_001330.1,NP_039595.1,75,1,22.46252,22.46221,0.44551,True,TMHMM2.0,33.0,75.0,TMHMM2.0,10.0,32.0,TMHMM2.0,1.0,9.0,RefSeq
1,NC_001331.1,NP_039601.1,30,1,19.48607,19.48607,0.86987,True,TMHMM2.0,1.0,6.0,TMHMM2.0,7.0,29.0,TMHMM2.0,30.0,30.0,RefSeq
2,NC_001331.1,NP_039602.1,83,1,23.01476,5.44076,0.03037,NaN,TMHMM2.0,80.0,83.0,TMHMM2.0,57.0,79.0,TMHMM2.0,1.0,56.0,RefSeq
3,NC_001331.1,NP_039603.1,82,2,43.70290,25.09221,0.99660,True,TMHMM2.0,80.0,82.0,TMHMM2.0,57.0,79.0,TMHMM2.0,43.0,56.0,RefSeq
4,NC_001331.1,NP_039604.1,437,2,37.26104,19.30140,0.90531,True,TMHMM2.0,436.0,437.0,TMHMM2.0,418.0,435.0,TMHMM2.0,27.0,417.0,RefSeq
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2820227,biochar_6175,biochar_6175_6,52,2,36.49023,36.49023,0.34638,True,TMHMM2.0,24.0,29.0,TMHMM2.0,30.0,51.0,TMHMM2.0,52.0,52.0,STV
2820228,biochar_6175,biochar_6175_10,222,1,22.77331,22.75221,0.84202,True,TMHMM2.0,1.0,6.0,TMHMM2.0,7.0,29.0,TMHMM2.0,30.0,222.0,STV
2820229,biochar_6175,biochar_6175_14,106,1,19.39243,19.39099,0.54127,True,TMHMM2.0,1.0,4.0,TMHMM2.0,5.0,24.0,TMHMM2.0,25.0,106.0,STV
2820230,biochar_6180,biochar_6180_1,812,1,21.67516,21.66885,0.97053,True,TMHMM2.0,1.0,6.0,TMHMM2.0,7.0,29.0,TMHMM2.0,30.0,812.0,STV


In [14]:
phage_trna_tmrna_metadata

,t(m)RNA_ID,Source,t(m)RNA,Start,Stop,Strand,Length,Permuted,Sequence,Phage_ID,Phage_source,Source_DB
0,NC_001335.1-1-aragorn,aragorn,tRNA-Asn(gtt),4200.0,4272.0,forward,73,-,tgacgtgtagctcaatggcagagcgcccgactgttaatcgggtggt...,NC_001335.1,RefSeq,RefSeq
1,NC_001335.1-2-aragorn,aragorn,tRNA-Trp(cca),4350.0,4423.0,forward,74,-,gtacacgtagctcaattggtagagcagcggtctccaaagccgccgg...,NC_001335.1,RefSeq,RefSeq
2,NC_001335.1-3-aragorn,aragorn,tRNA-Gln(ctg),4479.0,4553.0,forward,75,-,tccccgttcgtctaatcggtaagacacccggctctggaccgggcaa...,NC_001335.1,RefSeq,RefSeq
3,NC_001835.1-1-aragorn,aragorn,tRNA-Trp(cca),27504.0,27576.0,forward,73,-,tgcgagcatagtataatggtaatgctacagattccaaacctgtaaa...,NC_001835.1,RefSeq,RefSeq
4,NC_001835.1-2-aragorn,aragorn,tRNA-?(Stop|Leu)(ta),27644.0,27718.0,forward,75,-,tcacgttatagcttagggaagagcagttctgtttaggcggaagcaa...,NC_001835.1,RefSeq,RefSeq
...,...,...,...,...,...,...,...,...,...,...,...,...
1298176,biochar_6137-5-tRNAscanSE,tRNAscanSE,tRNA-Trp(cca),7766.0,7837.0,reverse,72,-,tgagtgtgctaacggcaagctgctggcctccaaagccatgcgatct...,biochar_6137,STV,STV
1298177,biochar_6137-6-tRNAscanSE,tRNAscanSE,tRNA-Ile2(cat),9150.0,9224.0,reverse,75,-,gggtccatagttcagtccggttagaacagtgacctcataagtcatt...,biochar_6137,STV,STV
1298178,biochar_6150-1-tRNAscanSE,tRNAscanSE,tRNA-Leu(taa),6817.0,6901.0,reverse,85,-,tggtgcccaaggggggactcgaacccccacagattgctccgctgaa...,biochar_6150,STV,STV
1298179,biochar_6150-2-tRNAscanSE,tRNAscanSE,tRNA-Ile2(cat),6653.0,6728.0,reverse,76,-,tggtgggtctgtaaggattcgaaccttcttctagcgattatgagtc...,biochar_6150,STV,STV


In [15]:
# CHVD, IGVD and GOV2 seem to have problems with an invertion of Phage_ID 
phage_virulent_factor_metadata

,Protein_ID,Aligned_Protein_in_VFDB,Phage_ID,Phage_source,Source_DB
0,YP_001449278.1,VFG001829(gb|WP_000752026),NC_004813.1,RefSeq,RefSeq
1,YP_009830126.1,VFG004787(gb|WP_000919350),NC_048634.1,RefSeq,RefSeq
2,YP_004286238.1,VFG000110(gb|WP_000979342),NC_015209.1,RefSeq,RefSeq
3,YP_009909342.1,VFG035045(gb|WP_001121226),NC_049944.1,RefSeq,RefSeq
4,NP_795658.1,VFG000951(gb|WP_011054794),NC_004588.1,RefSeq,RefSeq
...,...,...,...,...,...
23759,OTU_1211_46,VFG012939(gb|WP_000703656),OTU_1211,IGVD,IGVD
23760,OTU_1796_17,VFG006815(gb|WP_010959001),OTU_1796,IGVD,IGVD
23761,OTU_1208_17,VFG012939(gb|WP_000703656),OTU_1208,IGVD,IGVD
23762,OTU_3600_1,VFG006815(gb|WP_010959001),OTU_3600,IGVD,IGVD


In [16]:
transcription_terminator_metadata

,Phage_ID,Terminator,Start,Stop,Sense,Loc,Confidence,Source_DB,Phage_source
0,J02448.1,TERM_1,1541,1556,+,F,100,Genbank,NaN
1,AF227250.1,TERM_1,1708,1724,+,F,100,Genbank,NaN
2,AF323671.1,TERM_1,2769,2792,+,F,100,Genbank,NaN
3,AF323671.1,TERM_2,4486,4497,+,F,95,Genbank,NaN
4,AF323671.1,TERM_3,4512,4526,+,F,85,Genbank,NaN
...,...,...,...,...,...,...,...,...,...
4229111,biochar_6173,TERM_1,3204,3231,+,F,84,STV,STV
4229112,biochar_6175,TERM_1,5213,5251,+,F,93,STV,STV
4229113,biochar_6175,TERM_2,8620,8650,+,F,93,STV,STV
4229114,biochar_6181,TERM_1,905,953,+,F,100,STV,STV
